# RAW DATA AD-HOC WRANGLER
1. Data Wrangling Briefing
2. Data Wrangling Setup
3. Wrangling Data Tables of Instagram 
4. Wrangling Data Tables of TikTok
5. Wrangling Data Charts of Instagram and TikTok

## 1. Data Wrangling Briefing:
During the evaluation from viewing the dataframes and screenshots of the original data, it was possible to gather the following details.

Wrangling Tasks:
- Eliminate datapoints that contain only the character "|_|"
- Remove NaN datapoints
- Transform one column into several using "|_|" as a delimiter.
- Set appropriate column headers
- Transform all numbers into floats

Additionally, the following raw datasets are redundant or decontextualized, and should be disregarded for the next steps:
- raw_ig_table_1
- raw_ig_table_2
- raw_ig_table_3
- raw_ig_chart_2
- raw_tk_table_1
- raw_tk_table_2
- raw_tk_table_4
- raw_tk_table_5
- raw_tk_chart_3

## 2. Data Wrangling Setup:

In [ ]:
# Getting Started
import os
import sys
import sqlite3
from datetime import datetime

import pandas as pd

actl_dir = os.getcwd()
root_dir = os.path.dirname(actl_dir)
sys.path.append(root_dir)

from src.utils import(
    DataFrameVisualizer,
    get_raw_db_path)

view = DataFrameVisualizer.viewer

raw_db = get_raw_db_path(root_dir)
conn = sqlite3.connect(raw_db)

transformed_db = os.path.join(root_dir, 'database/02_transformed.db')

In [ ]:
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
tables = [table[0] for table in tables]

dfs = {'ig_table': [], 'ig_chart': [], 'tk_table': [], 'tk_chart': []}

for table in tables:
    for key in dfs.keys():
        if f'raw_{key}' in table:
            dfs[key].append(pd.read_sql_query(f'SELECT * FROM {table}', conn))

ig_table_df = pd.concat(dfs['ig_table'], ignore_index=True)
ig_chart_df = pd.concat(dfs['ig_chart'], ignore_index=True)
tk_table_df = pd.concat(dfs['tk_table'], ignore_index=True)
tk_chart_df = pd.concat(dfs['tk_chart'], ignore_index=True)

raw_ig_table_0 = ig_table_df.iloc[:, [1]].copy()
raw_ig_table_4 = ig_table_df.iloc[:, [5]].copy()
raw_ig_table_5 = ig_table_df.iloc[:, [6]].copy()
raw_ig_table_6 = ig_table_df.iloc[:, [7]].copy()
raw_ig_chart_0 = ig_chart_df.iloc[:, [1]].copy()
raw_ig_chart_1 = ig_chart_df.iloc[:, [2]].copy()

raw_tk_table_0 = tk_table_df.iloc[:, [1]].copy()
raw_tk_table_3 = tk_table_df.iloc[:, [4]].copy()
raw_tk_table_6 = tk_table_df.iloc[:, [7]].copy()
raw_tk_table_7 = tk_table_df.iloc[:, [8]].copy()
raw_tk_chart_0 = tk_chart_df.iloc[:, [1]].copy()
raw_tk_chart_1 = tk_chart_df.iloc[:, [2]].copy()
raw_tk_chart_2 = tk_chart_df.iloc[:, [3]].copy()

ig_tables = [raw_ig_table_0, raw_ig_table_4, raw_ig_table_5, raw_ig_table_6]
ig_charts = [raw_ig_chart_0, raw_ig_chart_1]

tk_tables = [raw_tk_table_0, raw_tk_table_6, raw_tk_table_7]
tk_charts = [raw_tk_chart_0, raw_tk_chart_1, raw_tk_chart_2]

conn.close()

## 3. Wrangling Data Tables of Instagram:

In [ ]:
def split_values(val):
    return pd.Series(val.split('|_|', 1))

In [ ]:
for table in ig_tables:
    table.dropna(inplace=True)
    table[['category', 'value']] = table[table.columns[0]].apply(split_values)
    table.drop(table.columns[0], axis=1, inplace=True)
    table.drop(table.index[0], inplace=True)
    table.reset_index(drop=True, inplace=True)
raw_ig_table_4.rename(columns={'category': 'region'}, inplace=True)
raw_ig_table_5.rename(columns={'category': 'age_bk'}, inplace=True)
raw_ig_table_6.rename(columns={'category': 'gender'}, inplace=True) 

## 4. Wrangling Data Tables of TikTok:

In [ ]:
for table in tk_tables:
    table.dropna(inplace=True)
    table[['category', 'value']] = table[table.columns[0]].apply(split_values)
    if table['value'].str.contains('%').any():
        table['value'] = table['value'].str.split('%').str[0]
    table.drop(table.columns[0], axis=1, inplace=True)
    table.drop(table.index[0], inplace=True)
    table.reset_index(drop=True, inplace=True)
raw_tk_table_6.rename(columns={'category': 'age_bk'}, inplace=True)
raw_tk_table_7.rename(columns={'category': 'gender'}, inplace=True)

raw_tk_table_3.dropna(inplace=True)
raw_tk_table_3.reset_index(drop=True, inplace=True)
raw_tk_table_3.drop(raw_tk_table_3.index[0], inplace=True)
raw_tk_table_3 = raw_tk_table_3['raw_tk_table_3_1'].str.split(r'\|_\|', expand=True)
raw_tk_table_3.columns = ['year', 'APAC', 'NAMER', 'europe', 'LATAM', 'MENA']
for index, col in enumerate(raw_tk_table_3.columns):
    if index > 0:
        raw_tk_table_3[col] = raw_tk_table_3[col].astype('float64')
    else:
        raw_tk_table_3[col] = pd.to_datetime(raw_tk_table_3[col], format='%Y').dt.to_period('Y').dt.end_time
        raw_tk_table_3[col] = pd.to_datetime(raw_tk_table_3[col]).dt.date

In [ ]:
for index, (ig_table, tk_table) in enumerate(zip(ig_tables, tk_tables)):
    if index > 0:
        ig_table['value'] = ig_table['value'].astype('float64')
        tk_table['value'] = tk_table['value'].astype('float64')
raw_ig_table_6['value'] = raw_ig_table_6['value'].astype('float64')

## 5. Wrangling Data Charts of Instagram and TikTok

In [ ]:
for chart in ig_charts:
    chart.dropna(inplace=True)
    chart[['close_quarter', 'value(mm)']] = chart[chart.columns[0]].apply(split_values)
    chart.drop(chart.columns[0], axis=1, inplace=True)
    chart.drop(chart.index[0], inplace=True)
    chart.reset_index(drop=True, inplace=True)

for chart in tk_charts:
    chart.dropna(inplace=True)
    chart[['close_quarter', 'value(mm)']] = chart[chart.columns[0]].apply(split_values)
    chart.drop(chart.columns[0], axis=1, inplace=True)
    chart.drop(chart.index[0], inplace=True)
    chart.reset_index(drop=True, inplace=True)

In [ ]:
def quarter_str_to_datetime(data_str):
    parts = data_str.split()
    quarter = int(parts[0].replace('Q', ''))
    year = int(parts[1])

    close_quarts_dates = {
        1: datetime(year, 3, 31).date(),
        2: datetime(year, 6, 30).date(),
        3: datetime(year, 9, 30).date(),
        4: datetime(year, 12, 31).date()}
    return close_quarts_dates[quarter]

In [ ]:
for chart in ig_charts:
    chart['c_quarter'] = chart['close_quarter'].apply(quarter_str_to_datetime)
for chart in tk_charts:
    chart['c_quarter'] = chart['close_quarter'].apply(quarter_str_to_datetime)

In [ ]:
for index, (ig_chart, tk_chart) in enumerate(zip(ig_charts, tk_charts)):
    ig_chart['value(mm)'] = ig_chart['value(mm)'].astype('float64')
    tk_chart['value(mm)'] = tk_chart['value(mm)'].astype('float64')
raw_tk_chart_2['value(mm)'] = raw_tk_chart_2['value(mm)'].astype('float64')

In [ ]:
transf_conn = sqlite3.connect(transformed_db)
raw_ig_table_0.to_sql('ig_overview', transf_conn, if_exists='replace', index=False)
raw_ig_table_4.to_sql('ig_regions', transf_conn, if_exists='replace', index=False)
raw_ig_table_5.to_sql('ig_ages_bk', transf_conn, if_exists='replace', index=False)
raw_ig_table_6.to_sql('ig_genders', transf_conn, if_exists='replace', index=False)

raw_ig_chart_0.to_sql('ig_revs', transf_conn, if_exists='replace', index=False)
raw_ig_chart_1.to_sql('ig_MAUs', transf_conn, if_exists='replace', index=False)

raw_tk_table_0.to_sql('tk_overview', transf_conn, if_exists='replace', index=False)
raw_tk_table_3.to_sql('tk_regions', transf_conn, if_exists='replace', index=False)
raw_tk_table_6.to_sql('tk_ages_bk', transf_conn, if_exists='replace', index=False)
raw_tk_table_7.to_sql('tk_genders', transf_conn, if_exists='replace', index=False)

raw_tk_chart_0.to_sql('tk_revs', transf_conn, if_exists='replace', index=False)
raw_tk_chart_1.to_sql('tk_MAUs', transf_conn, if_exists='replace', index=False)
raw_tk_chart_2.to_sql('tk_downlds', transf_conn, if_exists='replace', index=False)
transf_conn.close()